# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_90_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_90_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004927,0.001350,0.993723,0.003345,0.002030,0.994625
1,0.996682,0.000414,0.002905,0.996100,0.000521,0.003379
2,0.003205,0.000699,0.996095,0.002819,0.000623,0.996558
3,0.004065,0.003263,0.992672,0.003906,0.002117,0.993978
4,0.996028,0.000021,0.003951,0.995773,0.000024,0.004204


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.010078,0.003001,0.986921,0.009916,0.003474,0.986609
1,0.457102,0.002879,0.540019,0.505986,0.002978,0.491037
2,0.998144,0.001284,0.000572,0.996786,0.002227,0.000987
3,0.988863,0.000017,0.011120,0.989342,0.000024,0.010634
4,0.006397,0.002261,0.991343,0.006481,0.001816,0.991703


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=3000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:25:14,573] A new study created in memory with name: no-name-022348d8-1959-4af8-b4f2-deaf2f416eba
                                                                                                                                                                                                                   

[I 2026-07-05 15:25:14,751] Trial 9 finished with value: 0.8359954623317479 and parameters: {'weight_class_0': 1.0262250085934133, 'weight_class_1': 0.022692496322119775, 'weight_class_2': 1.2534272435067007}. Best is trial 9 with value: 0.8359954623317479.
[I 2026-07-05 15:25:14,752] Trial 0 finished with value: 0.9042902733663016 and parameters: {'weight_class_0': 0.005020549289867377, 'weight_class_1': 0.008812353068512116, 'weight_class_2': 0.7476651336456264}. Best is trial 0 with value: 0.9042902733663016.
[I 2026-07-05 15:25:14,755] Trial 8 finished with value: 0.6748710997601813 and parameters: {'weight_class_0': 0.0014659705345561474, 'weight_class_1': 3.173629663963162, 'weight_class_2': 0.4028570326792345}. Best is trial 0 with value: 0.9042902733663016.
[I 2026-07-05 15:25:14,756] Trial 7 finished with value: 0.873672663936965 and parameters: {'weight_class_0': 1.897607694264434, 'weight_class_1': 4.781876988587423, 'weight_class_2': 0.049401410073672894}. Best is trial 0 w

Best trial: 6. Best value: 0.949103:   1%|█▏                                                                                                                                     | 25/3000 [00:00<00:58, 51.13it/s]

[I 2026-07-05 15:25:14,937] Trial 12 finished with value: 0.8666488901783099 and parameters: {'weight_class_0': 0.12319742883265099, 'weight_class_1': 0.27707773646052486, 'weight_class_2': 0.0018023796172947418}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:14,944] Trial 16 finished with value: 0.9326818870819559 and parameters: {'weight_class_0': 0.13617121120002784, 'weight_class_1': 0.2830288436029976, 'weight_class_2': 9.252355719307543}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:14,950] Trial 15 finished with value: 0.9298280958903412 and parameters: {'weight_class_0': 0.0930291584857552, 'weight_class_1': 0.1897637018353295, 'weight_class_2': 7.768764192494065}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:14,958] Trial 14 finished with value: 0.9324039667169447 and parameters: {'weight_class_0': 0.11605241414152075, 'weight_class_1': 0.2533253940696394, 'weight_class_2': 9.014989313721887}. Best is trial 6 

[I 2026-07-05 15:25:15,129] Trial 27 finished with value: 0.6646629892096141 and parameters: {'weight_class_0': 0.001129210428206735, 'weight_class_1': 1.308547626800867, 'weight_class_2': 0.013367084370190263}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:15,147] Trial 26 finished with value: 0.7009190544190372 and parameters: {'weight_class_0': 0.001484594453220546, 'weight_class_1': 1.1046768529742093, 'weight_class_2': 0.013880036216438792}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:15,157] Trial 28 finished with value: 0.6637456224557635 and parameters: {'weight_class_0': 0.001090148816953155, 'weight_class_1': 1.1550289227316919, 'weight_class_2': 0.010056180724328363}. Best is trial 6 with value: 0.9491025903697642.
[I 2026-07-05 15:25:15,174] Trial 29 finished with value: 0.7148843895071536 and parameters: {'weight_class_0': 0.0011113257949376145, 'weight_class_1': 0.8038632626694467, 'weight_class_2': 0.2565193859970058}. Best i

...

[I 2026-07-05 15:26:35,221] Trial 2980 finished with value: 0.9494320466008368 and parameters: {'weight_class_0': 0.05075335968027407, 'weight_class_1': 0.6070176843165295, 'weight_class_2': 0.4155856844630468}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,231] Trial 2979 finished with value: 0.9489701759552799 and parameters: {'weight_class_0': 0.0643382658005128, 'weight_class_1': 1.3373757675264426, 'weight_class_2': 0.4324419696282983}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,267] Trial 2982 finished with value: 0.9487337965217396 and parameters: {'weight_class_0': 0.020497981314839805, 'weight_class_1': 0.6244687246777603, 'weight_class_2': 0.43107352324946857}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,268] Trial 2981 finished with value: 0.9463794015470084 and parameters: {'weight_class_0': 0.02186688287663638, 'weight_class_1': 1.4119063927936748, 'weight_class_2': 0.4325556243185375}. Best

Best trial: 1121. Best value: 0.949833: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 2995/3000 [01:21<00:00, 30.05it/s]

[I 2026-07-05 15:26:35,416] Trial 2986 finished with value: 0.9463850902186248 and parameters: {'weight_class_0': 0.02156685203646661, 'weight_class_1': 1.3882067987002398, 'weight_class_2': 0.42733800805750993}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,422] Trial 2987 finished with value: 0.9487155724529219 and parameters: {'weight_class_0': 0.021418889917313437, 'weight_class_1': 0.6396374233874943, 'weight_class_2': 0.45596657533108376}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,545] Trial 2990 finished with value: 0.9469335366551016 and parameters: {'weight_class_0': 0.0960085158445066, 'weight_class_1': 0.6082587872582177, 'weight_class_2': 0.4311309383851576}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,551] Trial 2989 finished with value: 0.9464498279907859 and parameters: {'weight_class_0': 0.021492236937033237, 'weight_class_1': 1.3656393684761718, 'weight_class_2': 0.4340433641393805}. Be

Best trial: 1121. Best value: 0.949833: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [01:21<00:00, 36.97it/s]

[I 2026-07-05 15:26:35,687] Trial 2996 finished with value: 0.9496470799240061 and parameters: {'weight_class_0': 0.04581544802148492, 'weight_class_1': 0.8322979390618235, 'weight_class_2': 0.614038286656948}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,703] Trial 2997 finished with value: 0.9497240156797141 and parameters: {'weight_class_0': 0.04813969599366262, 'weight_class_1': 0.7996590221237957, 'weight_class_2': 0.6106022103929949}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,718] Trial 2999 finished with value: 0.9497334988722282 and parameters: {'weight_class_0': 0.048702452159894306, 'weight_class_1': 0.793501204992805, 'weight_class_2': 0.6106078563186682}. Best is trial 1121 with value: 0.94983269980651.
[I 2026-07-05 15:26:35,722] Trial 2998 finished with value: 0.9495680998972248 and parameters: {'weight_class_0': 0.04581112284330318, 'weight_class_1': 0.7983860802755921, 'weight_class_2': 0.6319808519356177}. Best i

In [8]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [9]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [10]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.0.2_xgboost_submission.csv', index=False)

In [11]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
